# Notebook 01: Exploratory Data Analysis & Leakage Investigation
**Project:** COVID-CT Deep Learning Classifier (`covid-ct-dashboard`)


> [!WARNING]
> **EDUCATIONAL & PORTFOLIO PROJECT ONLY • NOT A MEDICAL DEVICE**  
> *This notebook, associated models, and data pipelines are strictly for research, algorithmic benchmarking, and portfolio demonstration. They are NOT certified by the FDA, CE, or any medical regulatory agency and must NEVER be used for clinical decision making, diagnosis, triage, or patient treatment.*


### Objectives:
1. Audit class distribution across Normal (`nCT`), COVID-19 Positive (`pCT`), and Non-informative (`NiCT`).
2. Inspect raw image dimensions, channels, and integrity (zero corrupt files check).
3. Detect exact duplicates using MD5 hashing and perceptual near-duplicates via difference hashing (`dHash`).
4. **Investigate patient/scan identifiers in filenames**: Analyze whether slice names contain patient tags, scan IDs, or purely sequential numbers.
5. Export dataset audit summary to `powerbi/data/dataset_summary.csv`.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# Ensure src is importable
PROJECT_ROOT = Path('.').resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_DIR, CLASS_NAMES, POWERBI_DATA_DIR
print('Project Root:', PROJECT_ROOT)
print('Raw Data Path:', RAW_DATA_DIR)

## 1. Class Distribution & File Inventory

In [ ]:
inventory = []
for c in CLASS_NAMES:
    c_dir = RAW_DATA_DIR / c
    files = list(c_dir.glob('*.*')) if c_dir.exists() else []
    inventory.append({'class': c, 'count': len(files)})

inv_df = pd.DataFrame(inventory)
display(inv_df)

plt.figure(figsize=(7, 4), dpi=150)
plt.bar(inv_df['class'], inv_df['count'], color=['#14B8A6', '#3B82F6', '#F59E0B'])
plt.title('Raw CT Slice Class Distribution', fontweight='bold')
plt.ylabel('Number of Slices')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

## 2. Filename Structure & Patient ID Investigation
A critical vulnerability in slice-based medical imaging benchmarks is **data leakage**: if multiple slices from the same patient scan are randomly assigned across train and test sets, the model can memorize unique patient anatomy, yielding artificially elevated test scores.

Let's inspect sample filenames from each class:

In [ ]:
for c in CLASS_NAMES:
    c_dir = RAW_DATA_DIR / c
    sample_fnames = [f.name for f in sorted(list(c_dir.glob('*.*')))[:8]]
    print(f'{c}:', sample_fnames)

### 🔍 Key Finding on Filename Provenance:
- The Kaggle dataset (`azaemon/preprocessed-ct-scans-for-covid19`) renames all original images to `<class><index>.jpg` (e.g. `nCT1.jpg`, `pCT45.jpg`, `NiCT120.jpg`).
- **No explicit patient ID string or DICOM metadata header is preserved** in these flattened JPEG filenames.
- **Consequence & Mitigation**: Rather than performing naive uniform random splitting (which guarantees slice leakage), we employ **proxy grouping**: consecutive sequence blocks and perceptual difference hash (`dHash`) clustering are used to group adjacent slices into pseudo-patient volumes. A strict `GroupShuffleSplit` (70/15/15) is then enforced so that clusters never cross splits.

## 3. Dataset Health Check: Corrupt Files & Duplicates

In [ ]:
summary_csv = POWERBI_DATA_DIR / 'dataset_summary.csv'
if summary_csv.exists():
    summary_df = pd.read_csv(summary_csv)
    display(summary_df)
else:
    print('dataset_summary.csv not generated yet.')

## 4. Visual Exemplars by Pathology Class

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(12, 3 * len(CLASS_NAMES)), dpi=150)
for r, c in enumerate(CLASS_NAMES):
    c_dir = RAW_DATA_DIR / c
    samples = sorted(list(c_dir.glob('*.*')))[:4]
    for col, s_path in enumerate(samples):
        img = Image.open(s_path)
        axes[r, col].imshow(img, cmap='bone')
        axes[r, col].set_title(f'{c} - {s_path.name}', fontsize=8)
        axes[r, col].axis('off')

plt.tight_layout()
plt.show()